# 01 — Quickstart

This notebook walks through the **minimal** Stitcher v0.2 pipeline on **synthetic data** so you can verify the install without real beamline data.

If you already have your own `.h5` files, you can skip the synthetic-data cell and feed the paths straight into `Stitcher(...)`.

In [ ]:
import os
import tempfile

import h5py
import numpy as np
import cupy as cp

from stitcher import Stitcher, Utilities

import matplotlib.pyplot as plt
%matplotlib inline

## 1. Generate fake `.h5` volumes

Two constant-intensity 3D volumes — good enough to exercise the pipeline.

In [ ]:
tmp = tempfile.mkdtemp(prefix="stitcher_demo_")
left_path  = os.path.join(tmp, "left.h5")
right_path = os.path.join(tmp, "right.h5")

shape = (16, 128, 128)
for path, value in [(left_path, 100), (right_path, 200)]:
    with h5py.File(path, "w") as fh:
        fh.create_dataset("image", data=np.full(shape, value, dtype=np.uint16))

print("left :", left_path)
print("right:", right_path)

## 2. Build the `Stitcher`

Tell the stitcher where the files are, where the motors were, and what `mm_per_voxel` to use.

In [ ]:
motor_positions = np.array([
    [0.0, 0.0, 0.0],
    [0.5, 0.0, 0.0],   # right is shifted +0.5 mm in x
])

st = Stitcher(
    file_path_list=[left_path, right_path],
    physical_coordinates=motor_positions,
    mm_per_voxel=0.01,
    x_y_z_correspondance=(1, 2, 3),
    saving_path=os.path.join(tmp, "out"),
)

## 3. Organize, register, accumulate

In [ ]:
st.get_layers_in_z(tolerance_mm=2)
st.get_padding()
st.get_intersections()
st.check_padding(layer_index=0)   # visual sanity check

In [ ]:
st.compute_shift_in_layers(
    start_slice=st.img_depth // 2 - 2,
    end_slice=st.img_depth // 2 + 2,
    mask=True, mask_radius=50,
    downscale=2, downscale_stages=2,
    apply_affine_warp=True, keep_rigid_only=True,
)

st.get_displacement_pyramid(starting_coord=(0, 0, 0))
st.accumulate_displacement(exclude_NCC=50, weighted_avg=False, affine_operator=True)
st.compose_final_displacements()

## 4. Stitch and inspect

In [ ]:
st.push_stitch_parameters()
st.stitch_layers(chunk_size_series=8, chunk_size_parallel=2, n_cores=4)

out_file = os.path.join(st.saving_path, "Stitched_layers", "Layer_0.h5")
with h5py.File(out_file, "r") as fh:
    ds = fh["/stitched_data/stitched_image"]
    print("stitched shape:", ds.shape, "dtype:", ds.dtype)
    mid = ds.shape[0] // 2
    plt.imshow(ds[mid], cmap="gray")
    plt.title("Stitched middle slice")
    plt.colorbar()
    plt.show()

## 5. Next steps

* Replace the synthetic volumes with your real `.h5` files.
* Tune `downscale`, `mask_radius`, `apply_affine_warp`, … (see `docs/quickstart.md`).
* For rotation-stage data, see `02_full_pipeline.ipynb` and `03_stitching_with_rotation.ipynb`.